# Creating Vanilla Rainbow Tables

For N = 2**16, currently only making one table

## Imports

In [20]:
import pickle
import random
from hashlib import sha256
from tqdm import tqdm
from math import pi, sqrt, e, log

## Table Parameters

In [21]:
# initialise startpoints - either generate them or load from pickle - to keep same across runs
def get_startpoints(N, m_0, nlabel, alpha):
    # try opening pickle file, else generate and save
    try:
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'rb') as f:
            startpoints = pickle.load(f)

    # no file found - generate and save
    except FileNotFoundError:
        # random but unique - store as a set?
        startpoints = set()
        while len(startpoints) < m_0:
            startpoints.add(random.randint(0, N-1))

        # store startpoints in pickle file
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'wb') as f:
            pickle.dump(startpoints, f)

    # return the startpoints
    return startpoints

In [22]:
# label N to find easier - label is the exponent
nlabel = 16
##############################################################################
N = 2 ** 16 # keyspace
p = 1 - e ** -2 # our table coverage - 86%
##############################################################################
t = round(log(1-p)/log(1-N**(-1/3))) # chain length t
alpha = 0.95 # maximality factor
mt_target = N**(2/3) # our target mt
m_0 = round(mt_target/(1-alpha))    # m_0 - number of startpoints
##############################################################################
# initialise startpoints 
startpoints = get_startpoints(N, m_0, nlabel, alpha)


## Hash and Reduction Functions

In [23]:
# Hash function
def H(x):
	return int.from_bytes(sha256(x.to_bytes(8)).digest())

# Reduction function
# currently mod but should change to murmurhash in future
def r(y, i, ell=0):   # also takes in ell - number of tables - for future use (but currently ell=0)
	return (y + i + ell*t) % N

## Building Vanilla Table

In [24]:
# take in m_0 and t as parameters - how many chains to start with and how long to make the chains
# store the table as a dictionary of endpoint:startpoint pairs (rather than sp:ep for easier lookup later)
# store in a pickle file

def build_vanilla_table(t, alpha, startpoints):
    # store table in dictionary
    table = {}

    # for each startpoint
    for sp in tqdm(startpoints):
        current_point = sp  # keep track of current point in chain -  we want to store startpoint later

        # create the chain
        for i in range(t):
            # hash then reduce the value
            current_point = r(H(current_point), i)

        # check if there wasn't a chain merge (not in a value stored already) - if not then store in table
        if current_point not in table:
            table[current_point] = sp

    # store the table as a pickle file
    with open(f'vanilla_table_alpha_{alpha}_t_{t}.pkl', 'wb') as f:
        pickle.dump(table, f)

    return table

## Searching the Table

### Function to help hash and reduce accordingly to continue search

In [25]:
# function to continue search
# we need to take in what column we're at (c), our key (y), # columns (t), current total of hashes and reductions
def continue_search(y, t, c, hashes, reductions):
    # reduce c by 1 to move to the previous column
    c -= 1
    # number of columns between current column and end 
    # diff = t - c  
    # reduce y by the new c index
    x = r(y, c)
    reductions += 1
    # then hash and reduce however many times to move back through columns
    for i in range((t - c) - 1, 0, -1):    # decrease difference by 1 (as we already reduced by current diff index) then continuously reduce by 1
        x = r(H(x), t-i)
        hashes += 1
        reductions += 1

    # return x, c, hashes, reductions
    return x, c, hashes, reductions

### Searching vanilla table function

In [26]:
def search_vanilla_table(y, t, table):
    # keep track of hashes and reductions
    col_round = 0
    hashes = 0
    reductions = 0

    # keep track of column we're in 
    c = t - 1
    # reduce (r_t-1) the hash then compare in the table 
    x = r(y, c)
    reductions += 1

    # while we haven't reached the end of our chain
    while c > -1:
        # if there is a match in the keys (our endpoints), regenerate chain until we find the key
        if (x in table.keys()):
            point = table[x]   # search for endpoint in table and get startpoint
            # rememeber to delete this
            sp = point
            # regenerate chain until column c - we are now in the column before the match
            for i in range(c):
                point = r(H(point), i)
                hashes += 1
                reductions += 1
            
            # hash the point - if it is a match we have found our preimage 
            if H(point) == y:
                print(sp)
                hashes += 1
                return point, hashes, reductions, col_round
            
            # if that didn't work then we ran into a false alarm
            else:
                # print("false alarm")
                # continue the search
                x, c, hashes, reductions = continue_search(y, t, c, hashes, reductions)
 
        # if we didn't find a match in endpoints, we need to restart the search
        else:
            # hash and reduce the ciphertext accordingly
            x, c, hashes, reductions = continue_search(y, t, c, hashes, reductions)

        col_round += 1

    # We have searched all columns - return -1
    return -1, hashes, reductions, col_round

## Run

### Precomputation Phase - Build the Table

In [27]:
# either build or load table
def get_vanilla_table():
    # try loading table from pickle file
    try:
        with open(f'vanilla_table_alpha_{alpha}_t_{t}.pkl', 'rb') as f:
            table = pickle.load(f)

    # if no pickle file found, build the table
    except FileNotFoundError:
        table = build_vanilla_table(t, alpha, startpoints)

    return table

In [28]:
vanilla = get_vanilla_table()

### Online Phase - Using the Table

In [29]:
# test column 74 to see why hashes and reductions are not what we think they are 
    # are we getting crazy numbers because when we encounter a false alarm we have to regenerate the chain to then check the hash, then because there isn't a match we have to continue the search?
    # we don't just get a difference of 1 between hashes and reductions, because even though we don't encounter any false alarms, if we have to shift a 
        # column, we add an extra reduction, so as we search the columns decreasingly, the difference between the two increases by t - c
    # we can test this by searching for an endpoint and seeing if we get t(t-1)/2
        # we get 4622 hashes
        # t (t - 1) / 2 = 80 (80 - 1)/ 2 = (80 * 79)/2 = 3160 calculations, not 4622
            # that's a 1462 difference 
y = H(42934)
value, hashes, reductions, col_round = search_vanilla_table(y, t, vanilla)
print(value, hashes, reductions, col_round)

-1 4909 4990 80


In [30]:
# get chain from pickle file
with open(f'vanilla_chain_alpha_{alpha}_t_{t}.pkl', 'rb') as f:
    chain = pickle.load(f)

# test each column in the chains
for i in range(1, len(chain)):  # i = 1 to 79
    print(f"Column {t - i}:")       # 
    print(f"     key = {chain[-(1+i)]}")
    y = H(chain[-(1+i)])
    value, hashes, reductions, col_round = search_vanilla_table(y, t, vanilla)
    print(f"     {value}")
    print(f'     Hashes: {hashes}, Reductions: {reductions}')

Column 79:
     key = 3745
0
     3745
     Hashes: 80, Reductions: 80
Column 78:
     key = 8251
0
     8251
     Hashes: 80, Reductions: 81
Column 77:
     key = 41276
0
     41276
     Hashes: 81, Reductions: 83
Column 76:
     key = 9819
0
     9819
     Hashes: 83, Reductions: 86
Column 75:
     key = 281
0
     281
     Hashes: 86, Reductions: 90
Column 74:
     key = 40959
0
     40959
     Hashes: 90, Reductions: 95
Column 73:
     key = 16034
4291
     16034
     Hashes: 83, Reductions: 86
Column 72:
     key = 39210
0
     39210
     Hashes: 254, Reductions: 261
Column 71:
     key = 49034
0
     49034
     Hashes: 181, Reductions: 189
Column 70:
     key = 50
6000
     50
     Hashes: 80, Reductions: 80
Column 69:
     key = 41036
528
     41036
     Hashes: 189, Reductions: 198
Column 68:
     key = 48002
0
     48002
     Hashes: 283, Reductions: 294
Column 67:
     key = 29090
0
     29090
     Hashes: 216, Reductions: 228
Column 66:
     key = 2033
0
     2033
     Hashe

In [31]:
# see if other elements not in the table produce the same number of hashes and reductions
y = H(3614213)
value, hashes, reductions, col_round = search_vanilla_table(y, t, vanilla)
print(value, hashes, reductions, col_round)

-1 4807 4888 80


In [32]:
y = H(89760)
value, hashes, reductions, col_round = search_vanilla_table(y, t, vanilla)
print(value, hashes, reductions, col_round)

-1 4922 5003 80


In [33]:
y = H(2)
value, hashes, reductions, col_round = search_vanilla_table(y, t, vanilla)
print(value, hashes, reductions, col_round)

2
2 4996 5075 79


In [34]:
vanilla

{45193: 0,
 49890: 1,
 37358: 2,
 4008: 5,
 29606: 6,
 42822: 9,
 25293: 10,
 36277: 14,
 27282: 15,
 21696: 16,
 33366: 22,
 18897: 24,
 10645: 25,
 45450: 27,
 14827: 28,
 21897: 29,
 727: 30,
 47396: 31,
 12744: 33,
 6277: 36,
 9277: 38,
 27436: 39,
 26960: 41,
 2723: 45,
 44953: 48,
 14135: 49,
 60670: 50,
 2561: 59,
 34029: 61,
 65438: 62,
 18586: 64,
 13920: 65,
 27459: 67,
 43357: 68,
 8116: 71,
 14547: 78,
 7332: 80,
 54302: 84,
 48596: 85,
 46477: 86,
 47390: 87,
 2682: 88,
 18951: 90,
 52178: 91,
 22791: 92,
 54703: 94,
 2465: 95,
 54273: 96,
 55307: 98,
 11250: 99,
 51112: 102,
 47202: 105,
 32506: 107,
 65355: 108,
 51974: 111,
 34438: 112,
 6483: 113,
 36437: 117,
 32947: 120,
 45755: 124,
 30879: 128,
 12352: 130,
 15004: 137,
 21630: 141,
 39612: 143,
 44498: 144,
 22005: 146,
 38010: 147,
 52874: 149,
 18247: 150,
 23443: 152,
 26204: 153,
 38460: 156,
 29649: 161,
 11866: 167,
 4057: 169,
 47001: 170,
 48200: 171,
 27629: 172,
 64897: 173,
 42334: 177,
 29981: 179,
 45

In [35]:
chain

[0,
 15868,
 911,
 46565,
 28284,
 12652,
 29453,
 65193,
 6243,
 49082,
 29768,
 27918,
 23871,
 31579,
 13413,
 2743,
 45695,
 62846,
 12541,
 54922,
 33018,
 30409,
 60587,
 45861,
 16707,
 60331,
 9859,
 15911,
 13065,
 55650,
 33634,
 11575,
 59649,
 34502,
 10762,
 5735,
 13535,
 22621,
 51566,
 3840,
 35223,
 5543,
 59431,
 18793,
 32230,
 13368,
 18706,
 44759,
 14557,
 12531,
 24631,
 56644,
 13251,
 51970,
 46547,
 42621,
 8869,
 37971,
 41509,
 34905,
 38502,
 13546,
 61315,
 7898,
 14726,
 6688,
 2033,
 29090,
 48002,
 41036,
 50,
 49034,
 39210,
 16034,
 40959,
 281,
 9819,
 41276,
 8251,
 3745,
 45193]

In [36]:
# regenerate chain
remade_chain = []
p = 0
remade_chain.append(p)

for i in range(t):
    # hash then reduce
    p = r(H(p), i)
    # store point in chain
    remade_chain.append(p)


